# MLP Classifier: Underfitting, Overfitting, Regularization & Generalization

Demonstration using synthetic classification dataset

## Import Libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.neural_network import  MLPClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, accuracy_score
from sklearn.preprocessing import StandardScaler

## Generate Dataset

In [ ]:
X_class, y_class = make_classification(n_samples=300, n_features=2, n_redundant=0, 
                                        n_informative=2, n_clusters_per_class=1, 
                                        random_state=42)

# Split and scale
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(
    X_class, y_class, test_size=0.3, random_state=42
)

scaler = StandardScaler()
X_train_c = scaler.fit_transform(X_train_c)
X_test_c = scaler.transform(X_test_c)

print(f"Training samples: {len(X_train_c)}")
print(f"Test samples: {len(X_test_c)}")

## Part 1: Underfitting, Overfitting & Good Fit

In [ ]:
# Underfitting
clf_underfit = MLPClassifier(hidden_layer_sizes=(2,), max_iter=1000, random_state=42)
clf_underfit.fit(X_train_c, y_train_c)

# Overfitting
clf_overfit = MLPClassifier(hidden_layer_sizes=(100, 100, 100), max_iter=2000, 
                            alpha=0.0, random_state=42)
clf_overfit.fit(X_train_c, y_train_c)

# Good fit
clf_goodfit = MLPClassifier(hidden_layer_sizes=(20, 10), max_iter=1000, 
                            alpha=0.01, early_stopping=True, random_state=42)
clf_goodfit.fit(X_train_c, y_train_c)

# Results
print("Classification Results:\n")
for name, model in [('Underfitting', clf_underfit), 
                     ('Overfitting', clf_overfit), 
                     ('Good Fit', clf_goodfit)]:
    train_acc = accuracy_score(y_train_c, model.predict(X_train_c))
    test_acc = accuracy_score(y_test_c, model.predict(X_test_c))
    print(f"{name}:")
    print(f"  Train Accuracy: {train_acc:.4f}")
    print(f"  Test Accuracy:  {test_acc:.4f}")
    print(f"  Gap: {abs(train_acc - test_acc):.4f}\n")

## Visualize Decision Boundaries

In [ ]:
def plot_decision_boundary(model, X, y, title, ax):
    h = 0.02
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                         np.arange(y_min, y_max, h))
    
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)
    
    ax.contourf(xx, yy, Z, alpha=0.3, cmap='RdYlBu')
    ax.scatter(X[:, 0], X[:, 1], c=y, edgecolors='k', cmap='RdYlBu', s=40)
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel('Feature 1')
    ax.set_ylabel('Feature 2')

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

plot_decision_boundary(clf_underfit, X_test_c, y_test_c, 'Underfitting', axes[0])
plot_decision_boundary(clf_overfit, X_test_c, y_test_c, 'Overfitting', axes[1])
plot_decision_boundary(clf_goodfit, X_test_c, y_test_c, 'Good Fit', axes[2])

plt.tight_layout()
plt.show()

## Part 2: Regularization Techniques

### 2.1 L2 Regularization

In [ ]:
# Test different alpha values
alphas = [0.0, 0.0001, 0.001, 0.01, 0.1, 1.0]

print("="*70)
print("L2 REGULARIZATION EFFECT")
print("="*70)

l2_results = []
for alpha in alphas:
    clf = MLPClassifier(
        hidden_layer_sizes=(50, 50),
        alpha=alpha,
        max_iter=1000,
        random_state=42
    )
    clf.fit(X_train_c, y_train_c)
    
    train_acc = accuracy_score(y_train_c, clf.predict(X_train_c))
    test_acc = accuracy_score(y_test_c, clf.predict(X_test_c))
    gap = abs(train_acc - test_acc)
    
    l2_results.append([alpha, train_acc, test_acc, gap])
    print(f"alpha={alpha:6.4f} | Train: {train_acc:.4f} | Test: {test_acc:.4f} | Gap: {gap:.4f}")

print("\n✓ Optimal alpha balances train and test performance")

In [ ]:
# Plot L2 effect
l2_results = np.array(l2_results)

plt.figure(figsize=(10, 6))
plt.plot(l2_results[:, 0], l2_results[:, 1], 'o-', label='Train Accuracy', linewidth=2, markersize=8)
plt.plot(l2_results[:, 0], l2_results[:, 2], 's-', label='Test Accuracy', linewidth=2, markersize=8)
plt.plot(l2_results[:, 0], l2_results[:, 3], '^-', label='Train-Test Gap', linewidth=2, markersize=8)
plt.xscale('log')
plt.xlabel('L2 Regularization (alpha)', fontsize=12)
plt.ylabel('Accuracy / Gap', fontsize=12)
plt.title('Effect of L2 Regularization', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 2.2 Early Stopping

In [ ]:
print("="*70)
print("EARLY STOPPING EFFECT")
print("="*70)

# Without early stopping
clf_no_early = MLPClassifier(
    hidden_layer_sizes=(50, 50),
    max_iter=2000,
    early_stopping=False,
    random_state=42
)
clf_no_early.fit(X_train_c, y_train_c)

# With early stopping
clf_early = MLPClassifier(
    hidden_layer_sizes=(50, 50),
    max_iter=2000,
    early_stopping=True,
    validation_fraction=0.2,
    n_iter_no_change=10,
    random_state=42
)
clf_early.fit(X_train_c, y_train_c)

print("\nWithout Early Stopping:")
print(f"  Iterations: {clf_no_early.n_iter_}")
print(f"  Train Acc:  {accuracy_score(y_train_c, clf_no_early.predict(X_train_c)):.4f}")
print(f"  Test Acc:   {accuracy_score(y_test_c, clf_no_early.predict(X_test_c)):.4f}")

print("\nWith Early Stopping:")
print(f"  Iterations: {clf_early.n_iter_}")
print(f"  Train Acc:  {accuracy_score(y_train_c, clf_early.predict(X_train_c)):.4f}")
print(f"  Test Acc:   {accuracy_score(y_test_c, clf_early.predict(X_test_c)):.4f}")

print(f"\n✓ Iterations saved: {clf_no_early.n_iter_ - clf_early.n_iter_}")
print("  Early stopping prevents overfitting and saves computation")

### 2.3 Model Complexity

In [ ]:
print("="*70)
print("MODEL COMPLEXITY EFFECT")
print("="*70)

architectures = [
    (5,),
    (10,),
    (25,),
    (50,),
    (25, 10),
    (50, 25),
    (100, 50),
    (100, 100)
]

complexity_results = []
for arch in architectures:
    clf = MLPClassifier(
        hidden_layer_sizes=arch,
        alpha=0.01,
        max_iter=1000,
        random_state=42
    )
    clf.fit(X_train_c, y_train_c)
    
    train_acc = accuracy_score(y_train_c, clf.predict(X_train_c))
    test_acc = accuracy_score(y_test_c, clf.predict(X_test_c))
    gap = abs(train_acc - test_acc)
    
    complexity_results.append([str(arch), train_acc, test_acc, gap])
    print(f"Architecture {str(arch):15} | Train: {train_acc:.4f} | Test: {test_acc:.4f} | Gap: {gap:.4f}")

print("\n✓ Moderate complexity with regularization works best")

## Part 3: Improving Generalization

### 3.1 More Training Data

In [ ]:
print("="*70)
print("EFFECT OF TRAINING DATA SIZE")
print("="*70)

sample_sizes = [50, 100, 200, 500, 1000]

data_results = []
for n_samples in sample_sizes:
    # Generate data
    X_temp, y_temp = make_classification(
        n_samples=n_samples, 
        n_features=2, 
        n_redundant=0,
        n_informative=2, 
        n_clusters_per_class=1,
        random_state=42
    )
    
    X_tr, X_te, y_tr, y_te = train_test_split(X_temp, y_temp, test_size=0.3, random_state=42)
    
    # Scale
    scaler_temp = StandardScaler()
    X_tr = scaler_temp.fit_transform(X_tr)
    X_te = scaler_temp.transform(X_te)
    
    # Train
    clf = MLPClassifier(
        hidden_layer_sizes=(50, 25),
        alpha=0.01,
        early_stopping=True,
        max_iter=1000,
        random_state=42
    )
    clf.fit(X_tr, y_tr)
    
    train_acc = accuracy_score(y_tr, clf.predict(X_tr))
    test_acc = accuracy_score(y_te, clf.predict(X_te))
    gap = abs(train_acc - test_acc)
    
    data_results.append([n_samples, train_acc, test_acc, gap])
    print(f"Samples: {n_samples:4} | Train: {train_acc:.4f} | Test: {test_acc:.4f} | Gap: {gap:.4f}")

print("\n✓ More training data improves generalization (smaller gap)")

In [ ]:
# Visualize effect of data size
data_results = np.array(data_results)

plt.figure(figsize=(10, 6))
plt.plot(data_results[:, 0], data_results[:, 1], 'o-', label='Train Accuracy', linewidth=2, markersize=8)
plt.plot(data_results[:, 0], data_results[:, 2], 's-', label='Test Accuracy', linewidth=2, markersize=8)
plt.plot(data_results[:, 0], data_results[:, 3], '^-', label='Train-Test Gap', linewidth=2, markersize=8)
plt.xlabel('Number of Training Samples', fontsize=12)
plt.ylabel('Accuracy / Gap', fontsize=12)
plt.title('Effect of Training Data Size on Generalization', fontsize=14, fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 3.2 Cross-Validation

In [ ]:
from sklearn.model_selection import cross_val_score

print("="*70)
print("CROSS-VALIDATION FOR RELIABLE EVALUATION")
print("="*70)

# Single train-test split
clf_single = MLPClassifier(
    hidden_layer_sizes=(50, 25),
    alpha=0.01,
    early_stopping=True,
    max_iter=1000,
    random_state=42
)
clf_single.fit(X_train_c, y_train_c)
single_score = accuracy_score(y_test_c, clf_single.predict(X_test_c))

# 5-Fold Cross-Validation
clf_cv = MLPClassifier(
    hidden_layer_sizes=(50, 25),
    alpha=0.01,
    max_iter=1000,
    random_state=42
)
cv_scores = cross_val_score(clf_cv, X_class, y_class, cv=5, scoring='accuracy')

print(f"\nSingle Train-Test Split:")
print(f"  Test Accuracy: {single_score:.4f}")

print(f"\n5-Fold Cross-Validation:")
print(f"  Fold Scores: {cv_scores}")
print(f"  Mean:  {cv_scores.mean():.4f}")
print(f"  Std:   {cv_scores.std():.4f}")

print("\n✓ Cross-validation provides more reliable performance estimate")

### 3.3 Hyperparameter Tuning

In [ ]:
from sklearn.model_selection import GridSearchCV

print("="*70)
print("HYPERPARAMETER TUNING WITH GRID SEARCH")
print("="*70)

# Define parameter grid
param_grid = {
    'hidden_layer_sizes': [(20,), (50,), (20, 10), (50, 25)],
    'alpha': [0.0001, 0.001, 0.01, 0.1],
    'learning_rate_init': [0.001, 0.01]
}

# Grid search
clf_grid = MLPClassifier(
    max_iter=1000,
    early_stopping=True,
    random_state=42
)

grid_search = GridSearchCV(
    clf_grid,
    param_grid,
    cv=3,
    scoring='accuracy',
    n_jobs=-1,
    verbose=0
)

print("\nSearching for best hyperparameters...")
grid_search.fit(X_train_c, y_train_c)

print(f"\nBest Parameters: {grid_search.best_params_}")
print(f"Best CV Score:   {grid_search.best_score_:.4f}")
print(f"Test Accuracy:   {accuracy_score(y_test_c, grid_search.predict(X_test_c)):.4f}")

print("\n✓ Hyperparameter tuning optimizes generalization")

### 3.4 Ensemble Methods

In [ ]:
from scipy.stats import mode

print("="*70)
print("ENSEMBLE METHODS (VOTING)")
print("="*70)

# Train multiple models
n_models = 5
predictions = []

for i in range(n_models):
    clf = MLPClassifier(
        hidden_layer_sizes=(50, 25),
        alpha=0.01,
        early_stopping=True,
        max_iter=1000,
        random_state=i  # Different initialization
    )
    clf.fit(X_train_c, y_train_c)
    predictions.append(clf.predict(X_test_c))

# Majority voting
ensemble_pred = mode(predictions, axis=0, keepdims=False)[0]

# Single model
single_clf = MLPClassifier(
    hidden_layer_sizes=(50, 25),
    alpha=0.01,
    early_stopping=True,
    max_iter=1000,
    random_state=42
)
single_clf.fit(X_train_c, y_train_c)
single_pred = single_clf.predict(X_test_c)

print(f"\nSingle Model Accuracy: {accuracy_score(y_test_c, single_pred):.4f}")
print(f"Ensemble Accuracy:     {accuracy_score(y_test_c, ensemble_pred):.4f}")

print("\n✓ Ensemble of models provides more robust predictions")

### 3.5 Data Augmentation

In [ ]:
print("="*70)
print("DATA AUGMENTATION FOR SMALL DATASETS")
print("="*70)

# Small dataset
X_small, y_small = make_classification(
    n_samples=100,
    n_features=2,
    n_redundant=0,
    n_informative=2,
    n_clusters_per_class=1,
    random_state=42
)
X_tr_s, X_te_s, y_tr_s, y_te_s = train_test_split(X_small, y_small, test_size=0.3, random_state=42)

# Scale
scaler_s = StandardScaler()
X_tr_s_scaled = scaler_s.fit_transform(X_tr_s)
X_te_s_scaled = scaler_s.transform(X_te_s)

# Without augmentation
clf1 = MLPClassifier(hidden_layer_sizes=(20, 10), alpha=0.01, max_iter=1000, random_state=42)
clf1.fit(X_tr_s_scaled, y_tr_s)

# With augmentation (add noise)
X_augmented = np.vstack([X_tr_s_scaled, X_tr_s_scaled + np.random.normal(0, 0.1, X_tr_s_scaled.shape)])
y_augmented = np.hstack([y_tr_s, y_tr_s])

clf2 = MLPClassifier(hidden_layer_sizes=(20, 10), alpha=0.01, max_iter=1000, random_state=42)
clf2.fit(X_augmented, y_augmented)

print(f"\nOriginal Data Only:      {accuracy_score(y_te_s, clf1.predict(X_te_s_scaled)):.4f}")
print(f"With Augmentation:       {accuracy_score(y_te_s, clf2.predict(X_te_s_scaled)):.4f}")
print(f"\nTraining samples: {len(X_tr_s)} → {len(X_augmented)}")

print("\n✓ Data augmentation can improve generalization with limited data")

## Summary: Complete Comparison

In [ ]:
print("="*70)
print("FINAL COMPARISON: ALL TECHNIQUES")
print("="*70)

# Compare all approaches
approaches = {
    'Underfitting': clf_underfit,
    'Overfitting': clf_overfit,
    'Good Fit': clf_goodfit,
    'Grid Search Optimized': grid_search.best_estimator_
}

final_results = []
for name, clf in approaches.items():
    train_acc = accuracy_score(y_train_c, clf.predict(X_train_c))
    test_acc = accuracy_score(y_test_c, clf.predict(X_test_c))
    gap = abs(train_acc - test_acc)
    
    final_results.append([name, train_acc, test_acc, gap])
    print(f"{name:25} | Train: {train_acc:.4f} | Test: {test_acc:.4f} | Gap: {gap:.4f}")

print("\n" + "="*70)

In [ ]:
# Final visualization
final_results = np.array(final_results, dtype=object)
names = final_results[:, 0]
train_accs = final_results[:, 1].astype(float)
test_accs = final_results[:, 2].astype(float)
gaps = final_results[:, 3].astype(float)

x = np.arange(len(names))
width = 0.25

fig, ax = plt.subplots(figsize=(12, 6))
ax.bar(x - width, train_accs, width, label='Train Accuracy', alpha=0.8)
ax.bar(x, test_accs, width, label='Test Accuracy', alpha=0.8)
ax.bar(x + width, gaps, width, label='Train-Test Gap', alpha=0.8)

ax.set_ylabel('Accuracy / Gap', fontsize=12)
ax.set_title('Complete Comparison: Fitting Types & Generalization Techniques', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(names, rotation=20, ha='right')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## Key Takeaways

### Fitting Types:
- **Underfitting:** Model too simple → Both train & test errors high
- **Overfitting:** Model too complex → Train error low, test error high (large gap)
- **Good Fit:** Balanced model → Both errors low with small gap

### Regularization Techniques:
1. **L2 Regularization (alpha):** Penalizes large weights
2. **Early Stopping:** Stops when validation error increases
3. **Model Complexity:** Choose appropriate network size

### Improving Generalization:
1. **More Training Data:** Reduces overfitting and improves test performance
2. **Cross-Validation:** Provides reliable performance estimates
3. **Hyperparameter Tuning:** Finds optimal model configuration
4. **Ensemble Methods:** Combines multiple models for robustness
5. **Data Augmentation:** Expands limited datasets

### Best Practices:
- Always scale features (StandardScaler)
- Use L2 regularization by default
- Enable early stopping to prevent overtraining
- Monitor train-test gap to detect overfitting
- Use cross-validation for model evaluation
- Tune hyperparameters systematically